![interpreto_banner](../assets/img/interpreto_banner.png){ style="display:block; max-width:100%; height:auto; margin:0 auto;" }

# Generation Concept-based Explanation Tutorial

Welcome to this tutorial, our will be to obtain concept-based explanations starting from the beginning.

We will start with a minimal example in section 1 and then go through the key steps of the concept-based:

0. [➗ **Split** your model in two parts](#split)
1. [⏩ **Minimal** example: Top-k tokens for neurons](#minimal)
2. [🚦 Compute a dataset of **activations**](#activations)
3. [🏋️‍♂️ **Fit** a concept model on activations](#fit)
4. [🏷️ **Interpret** the concept dimensions](#interpret)

On which we add two bonus steps present in most papers:

5. [📍 **Local** concept analysis](#locally)
6. [⚖️ **Evaluate** concept-based explanations](#evaluate)

*Author: Antonin Poché*

## 0. ➗ **Split** your model in two parts <a class="anchor" id="split"></a>

Let's take a Qwen3-0.6B for both the minimal example and the detailed pipeline. But you can naturally use larger models.

Here we split at the 6 / 28 layers. But you can specify the module path to split at.

To split the model, we use [`interpreto.TextTokensSplitter`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/text_tokens_splitter/), which wraps the `transformers` causal language model and computes token activations at the specified `split_point`.

> ➡️ **Note**
>
> Interpreto's splitting based on [`nnsight`](https://github.com/ndif-team/nnsight), so depending on your use case, you might want to use `nnsight` directly.

In [1]:
from interpreto import TextTokensSplitter

# 1. load and split the generation model
splitter = TextTokensSplitter(
    "Qwen/Qwen3.5-0.8B",
    task="text-generation",
    split_point=5,  # split at the 6th layer
    device_map="cuda",
    batch_size=256,  # high for the minimal example where samples are a single token
)

## 1. ⏩ **Minimal** example: Top-k tokens for neurons <a class="anchor" id="minimal"></a>

In [2]:
from interpreto.concepts import NeuronsAsConcepts, TopKInputs

# 2. No dataset of activation is needed as we consider the latent space as the concept space

# 3. No training neither
# Use `NeuronsAsConcepts` to use the concept-based pipeline with neurons
concept_explainer = NeuronsAsConcepts(splitter)

# 4. Use `TopKInputs` to get the top-k tokens that maximally activate each neuron
method = TopKInputs(
    concept_explainer=concept_explainer,
    use_vocab=True,  # use the vocabulary of the model and test all tokens (~150k with Qwen-0.6B)
    k=10,  # get the top 10 tokens for each neuron
)
topk_tokens = method.interpret(
    concepts_indices=list(range(5)),  # interpret the five first neurons
)

# show some neurons' interpretations
for concept_idx, tokens in topk_tokens.items():
    print(f"Concept {concept_idx}: {list(tokens.keys())}")

del concept_explainer, method, topk_tokens

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Concept 0: ['<|fim_middle|>', 'ä¾Ŀæ³ķé¡»ç»ıæī¹åĩĨçļĦé¡¹çĽ®å¤ĸ', '(UIAlertAction', 'ĠÐ¿ÑĢÐ¸Ð½Ñĥ', 'æĪĸè®¸æľīåĪ«äºº', '/tinyos', 'ãģĶäºĨ', 'íĬ¸ë¦½ìĸ´ëĵľë°ĶìĿ´ìłĢ', 'erusform', 'elementGuidId']
Concept 1: ['public', 'private', 'ĉdef', 'ĉpublic', 'def', 'const', 'import', '<?', 'void', 'class']
Concept 2: ['æĽ´å¤ļç±»ä¼¼éĹ®é¢ĺ', 'Ġveranst', 'ĠMcA', 'ĠAssicur', 'Ġváº¯c', 'versicher', 'ĠÐºÐ¾ÑĢÐ¾Ð½Ð°Ð²Ð¸', 'Ġmiseric', 'Ġubezpie', 'à¹Ĥà¸£à¸ĩà¸ŀà¸¢à¸²à¸ļà¸²à¸¥']
Concept 3: ['<|fim_suffix|>', '.ImageTransparentColor', 'è®©æĽ´å¤ļè¯»èĢħæ¬£èµı', 'ä¸ºè½¬è½½ä½ľåĵģ', '.StObject', 'ĠÐ¿ÑĢÐ¸Ð½Ñĥ', 'æĪĸè®¸æľīåĪ«äºº', 'Ġà¸Ĥà¸¶', 'Ġdumpster', ',cljs']
Concept 4: ['Calculate', 'è®¡ç®Ĺ', 'Compute', 'å®ļä¹īä¸º', 'æķ°æį®ç»ĵæŀĦ', 'Ġsklearn', 'ä¸ĵåĪ©ä¿¡æģ¯', 'ĠSNMP', 'ĠminOccurs', 'ĠFPGA']


> ❓ **The concepts are not interpretable?**
>
> Well, this is surely due to superposition, this is why we use dictionary learning.
>
> Let's explore and solve this problem in the next sections.

## 2. 🚦 Compute a datasets of **activations** <a class="anchor" id="activations"></a>

We will use the [`IMDB`](https://huggingface.co/datasets/stanfordnlp/imdb) dataset to build a dataset of activations.

> ⚠️ **Warning**
>
> The dataset used has a considerable impact on the concept-space obtained. Hence, in practice, we recommend to use a subset of the model training set.
>
> The concept model we train be it SAEs or others, find patterns in the dataset of activations, which explains the dependence of the concepts found on the activations dataset.

> 🔥 **Tip**
>
> The larger the dataset the better. But at the cost of computation time to get activations.
>
> Hence, the best dataset for SAEs would be one where each token is only seen once. But this is harder in practice, and such pipeline is not covered in this tutorial.

[`interpreto.TextTokensSplitter.get_activations()`](https://for-sight-ai.github.io/interpreto/api/concepts/splitters/text_tokens_splitter/#interpreto.TextTokensSplitter.get_activations) returns flattened non-special token activations by default.

In [3]:
from datasets import load_dataset

# take the whole dataset, more samples leads to better results, but some methods do not support too big datasets
imdb = load_dataset("stanfordnlp/imdb")["train"]["text"][:10000]

# compute the non-special-token activations of the whole IMDB dataset
# activations are flattened between the n_sample and seq_len dimensions
# which leads us to more then 6 million tokens
# (n * l, d)
splitter.batch_size = 8
activations, _ = splitter.get_activations(
    inputs=imdb,
    tqdm_bar=True,
)

print(f"{activations.shape = }")

Computing activations: 100%|██████████| 1250/1250 [01:07<00:00, 18.64batch/s]


activations.shape = torch.Size([2956055, 1024])


## 3. 🏋️‍♂️ **Fit** a concept model on activations <a class="anchor" id="fit"></a>

Now we can fit a concept model on the activations. They exist more or less complex concept models. Here we use an SAE, so it is quite complex and has a lot more parameters than a simple concept model.

In particular, we use [`interpreto.concepts.BatchTopK`](https://for-sight-ai.github.io/interpreto/api/concepts/concept_spaces/sae/#interpreto.concepts.methods.BatchTopKSAEConcepts).

The `concept_explainer` wraps around both `splitter`, the model wrapper, and the `concept_model`.

> ➡️ **Note**
> 
> Most of our concept models and all the SAEs implementations depend on the [Overcomplete](https://kempnerinstitute.github.io/overcomplete/) library. It is a library that provides a lot of concept models and optimization methods for concept extraction. So if you want to go deeper on these, we suggest digging there.

In [4]:
import torch

from interpreto.concepts.methods.overcomplete import BatchTopKSAEConcepts, DeadNeuronsReanimationLoss

top_k_individual = 10
concept_model_batch_size = 512
epochs = 5

# instantiate the concept explainer with the splitted model
concept_explainer = BatchTopKSAEConcepts(
    splitter,
    nb_concepts=1000,
    device="cuda",
    top_k=top_k_individual * concept_model_batch_size,
)

# train the SAE on the activations
log = concept_explainer.fit(
    activations=activations,
    criterion=DeadNeuronsReanimationLoss,  # set an MSE loss with dead neurons reanimation
    optimizer_class=torch.optim.Adam,
    scheduler_class=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_kwargs={"T_max": epochs, "eta_min": 1e-6},
    lr=1e-3,
    nb_epochs=epochs,
    batch_size=concept_model_batch_size,
    monitoring=1,
)

Epoch[1/5], Loss: 0.0012, R2: 0.6759, L0: 10.0015, Dead Features: 0.0%, Time: 18.7431 seconds
Epoch[2/5], Loss: 0.0010, R2: 0.7327, L0: 10.0015, Dead Features: 0.0%, Time: 18.6438 seconds
Epoch[3/5], Loss: 0.0010, R2: 0.7428, L0: 10.0014, Dead Features: 0.0%, Time: 18.3499 seconds
Epoch[4/5], Loss: 0.0009, R2: 0.7494, L0: 10.0015, Dead Features: 0.0%, Time: 18.1612 seconds
Epoch[5/5], Loss: 0.0009, R2: 0.7523, L0: 10.0015, Dead Features: 0.0%, Time: 18.1643 seconds


## 4. 🏷️ **Interpret** the concept dimensions <a class="anchor" id="interpret"></a>

Once the concept directions are set, it is time to interpret them. We explore top-k examples and labels generated by the model already held by the splitter.

### 4.1 Interpret concepts via top-k tokens

Via [`interpreto.concepts.TopKInputs`](https://for-sight-ai.github.io/interpreto/api/concepts/interpretations/topk_inputs/#interpreto.concepts.interpretations.TopKInputs), it is possible to extract the top-k tokens that activate each concept the most.

In [5]:
from interpreto.concepts import TopKInputs

interpretation_method = TopKInputs(
    concept_explainer=concept_explainer,
    concept_encoding_batch_size=512 * concept_model_batch_size,
    k=10,  # get the top 10 tokens for each concept
)
interpretations = interpretation_method.interpret(
    inputs=imdb,
    latent_activations=activations,
    concepts_indices="all",  # interpret all concepts
)

# Clean Ġ for Qwen specifically
interpretations = {
    concept_idx: [token.lstrip("Ġ") for token in tokens.keys()] if tokens else None
    for concept_idx, tokens in interpretations.items()
}

for concept_idx, tokens in list(interpretations.items())[:10]:
    print(f"Concept {concept_idx}: {tokens if tokens else None}")

Concept 0: ['on', 'On', 'on', 'ON', 'On', '-on', '(on', 'THE', ',on', 'the']
Concept 1: ['DVD', 'y', 'acting', 'from', 'should', 'figur', 'fights', 'flies', 'would', 'video']
Concept 2: ['you', 'she', 'Ford', 'na', 'Channel', 'land', 'ower', 'they', 'wife', 'bie']
Concept 3: ['to', 'from', 'that', 'know', 'of', 'begins', 'job', "'t", 'living', '1']
Concept 4: ['well', 'well', 'Well', ',', 'Well', 'ell', 'Okay', 'WELL', '...', 'erm']
Concept 5: ['OK', '.k', 'okay', 'ok', 'fine', 'with', 'kej', 'ok', 'acceptable', 'right']
Concept 6: ['do', 'did', 'do', 'does', 'does', 'DO', 'DID', 'DOES', 'Did', 'did']
Concept 7: ['ie', 'days', 'of', 'for', 'Ethics', 'second', 'from', 'recipients', 'clicked', 'awarded']
Concept 8: ['award', 'ards', 'Oscar', 'Award', 'Awards', 'Oscars', 'awards', 'Prize', 'nomination', 'nominated']
Concept 9: ['reading', 'read', 'Read', 'book', 'reads', 'books', 'Reading', 'reader', 'Read', 'novels']


> ➡️ **Note**
> 
> While not the most interpretable concepts, this is still better than the minimal example.
>
> Let's improve interpretations and see what happens.

### 4.2 Interpret concepts with an LLM labels

Here we use [`interpreto.concepts.LLMLabels`](https://for-sight-ai.github.io/interpreto/api/concepts/interpretations/llm_labels/#interpreto.concepts.interpretations.LLMLabels), it uses LLM to label the concepts based on examples activating the concept.

> ℹ️ **Note**
>
> Qwen3-0.6B is capable enough to label these concepts, so we reuse the generation model already held by the splitter.

In [6]:
from interpreto.concepts import LLMLabels

interpretation_method = LLMLabels(
    concept_explainer=concept_explainer,
    llm_interface=None,  # Reuse the generation model held by the splitter.
    k_examples=20,  # number of examples in each concept
    k_context=5,  # number of tokens before and after the maximally activating one to give context
    concept_encoding_batch_size=concept_model_batch_size,
)

# compute the labels for an arbitrary subset of the concepts
interpretations = interpretation_method.interpret(
    inputs=imdb,
    latent_activations=activations,
    concepts_indices=list(range(10)),  # we could put `"all"`, but that would take longer
)

for concept_id, label in interpretations.items():
    print(f"Concept {concept_id}: {label if label is not None else 'None'}")

[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Concept 0: Ġpeople
Concept 1: DVD
Concept 2: you
Concept 3: Ġjob
Concept 4: well
Concept 5: OK
Concept 6: Ġdo
Concept 7: Razz
Concept 8: Oscar
Concept 9: reading


> ➡️ **Note 1**
>
> The labels highly depend on the system prompt provided to the LLM interface.
>
> For labels formulation to better align with what you expect, you can set the `system_prompt` argument of the `LLMLabels`.
>
> Here is the default system prompt:

```python
SYSTEM_PROMPT_WITH_CONTEXT = """You assign a short label to a concept from examples.

Each example contains highlighted text between `<< >>`. The highlighted text is the main evidence for the concept. Activation scores range from 0 to 10; higher scores provide stronger evidence.

Infer the most specific concept shared by the strongest examples.

Output exactly one noun phrase of 1 to 5 words.

Output only the label.
Do not explain your answer.
Do not summarize the examples.
Do not mention activations, examples, tokens, or markers.
Do not write a sentence.
Do not use quotes, bullets, Markdown, or a prefix such as "Label:".
"""
```

> ➡️ **Note 2**
>
> Having our concepts and their interpretation is great. But what is really useful is to know:
> - When are each concept used? (see [next section](#locally))
> - If the concepts are pertinent. (see [final section](#evaluate))

## 5. 📍 **Local** concept analysis <a class="anchor" id="locally"></a>

To see the important concepts in a sample, there are two complementary steps:
- Find the most important concepts for the predicted tokens.
- Highlight the input tokens activating this concept.

### 5.0 Create a random sample and decompose it into tokens

In [7]:
# create a sample
sample = ["Interpreto has awesome visualizations."]

# split text between tokens
sample_tokens = splitter.tokenizer.tokenize(sample[0])

# to include special tokens:
# encoded = splitter.tokenizer(sample[0], add_special_tokens=True)
# sample_tokens = splitter.tokenizer.convert_ids_to_tokens(encoded["input_ids"])

# specific to the model
sample_tokens = [tok.replace("Ġ", " ") for tok in sample_tokens]

print(f"The sample is: {sample[0]}\n\nIt is decomposed in {len(sample_tokens)} tokens:\n{sample_tokens}")

The sample is: Interpreto has awesome visualizations.

It is decomposed in 8 tokens:
['Inter', 'pre', 'to', ' has', ' awesome', ' visual', 'izations', '.']


### 5.1 Important concepts for text (with respect to the output)

Here we use the gradient of the concept-to-output function to estimate the importance of each concept for the outputs of the model.

[`interpreto.concepts.ConceptAutoEncoderExplainer.concept_output_gradient()`](https://for-sight-ai.github.io/interpreto/api/concepts/methods/base/#interpreto.concepts.ConceptAutoEncoderExplainer.concept_output_gradient)

In [8]:
# (seq_len (out), seq_len (in), nb_concepts)
local_importances = concept_explainer.concept_output_gradient(
    inputs=sample,
    targets=None,  # all predicted tokens
)[0]  # only one sample

# we take the sum over the input sequence dimension, has we focus on the concept-output relationship
# (seq_len (out), nb_concepts)
local_importances = local_importances.abs().sum(dim=1)
print(f"{local_importances.shape = }")

local_importances.shape = torch.Size([8, 1000])


### 5.2 Input tokens activating the concepts

This step just looks at the concepts activation on the input tokens. It is a simple way to check if the concepts are activated on the input tokens.

In [9]:
# compute the latent activations
# (seq_len, d_model)
local_activations, _ = splitter.get_activations(sample)

# and the concepts activations
# (seq_len, nb_concepts)
concepts_activations = concept_explainer.activations_to_concepts(local_activations)
print(f"{concepts_activations.shape = }")

concepts_activations.shape = torch.Size([8, 1000])


### 5.3 Visualization

We did not interpret all concepts previously, so we need to interpret the locally most important concepts.

In [10]:
# select the most important concepts for each output token
to_interpret = []
for importance in local_importances:
    top10 = torch.argsort(importance, descending=True)[:10]
    to_interpret.append(top10)

to_interpret = torch.concat(to_interpret, dim=0).unique()
to_interpret = [c.item() for c in to_interpret if c.item() not in interpretations.keys()]

# interpret these concepts
new_interpretations = interpretation_method.interpret(
    inputs=imdb,
    latent_activations=activations,
    concepts_indices=to_interpret,
)
interpretations.update(new_interpretations)

In [11]:
from interpreto import plot_concepts

plot_concepts(
    concepts_activations=concepts_activations,
    concepts_importances=local_importances,
    concepts_labels=interpretations,
    sample=sample_tokens,
)

> 🔥 **Tip**
>
> By default, the most important concepts for the whole sample are shown.
>
> But you can click on an output token to select it and look at concepts for that token only.

## 6. ⚖️ **Evaluate** concept-based explanations <a class="anchor" id="evaluate"></a>

Let's compare the minimal example and the Batch TopK SAE explainers.

In [12]:
test_inputs = load_dataset("stanfordnlp/imdb")["test"]["text"][:100]  # let's take one thousand test samples

# Compute the non-special-token activations
test_activations, _ = splitter.get_activations(
    inputs=test_inputs,
)

### 6.1 🌐 Evaluate the concept-space from the [third part](#fit)

> ⚠️ Warning:
>
> These metrics should only be used to compare the concept-space trained in similar contexts, same model, split point, activation dataset...

**Reconstruction error**

- [`interpreto.concepts.metrics.MSE`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/reconstruction_metrics/#interpreto.concepts.metrics.MSE)
- [`interpreto.concepts.metrics.FID`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/reconstruction_metrics/#interpreto.concepts.metrics.FID)

**Sparsity**

- [`interpreto.concepts.metrics.Sparsity`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/sparsity_metrics/#interpreto.concepts.metrics.Sparsity)
- [`interpreto.concepts.metrics.SparsityRatio`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/sparsity_metrics/#interpreto.concepts.metrics.SparsityRatio)

**Dictionary metrics**

- [`interpreto.concepts.metrics.Stability`](https://for-sight-ai.github.io/interpreto/api/concepts/metrics/dictionary_metrics/#interpreto.concepts.metrics.Stability)

> ➡️ **Note**
>
> We do not apply the stability metric here because it requires to retrain the concept space several times and compare them. It would be too long for the tutorial. Nonetheless, SAEs in general are not really stable.

In [13]:
from interpreto.concepts.metrics import FID, MSE, Sparsity, SparsityRatio

# BatchTopKSAEConcepts
mse = MSE(concept_explainer).compute(test_activations)
fid = FID(concept_explainer).compute(test_activations)
sparsity = Sparsity(concept_explainer).compute(test_activations)
ratio = SparsityRatio(concept_explainer).compute(test_activations)

print(f"MSE: {round(mse, 3)}, FID: {round(fid, 3)}, Sparsity: {round(sparsity, 3)}, Sparsity ratio: {round(ratio, 3)}")

# NeuronsAsConcepts
identity_explainer = NeuronsAsConcepts(splitter)
mse = MSE(identity_explainer).compute(test_activations)
fid = FID(identity_explainer).compute(test_activations)
sparsity = Sparsity(identity_explainer).compute(test_activations)
ratio = SparsityRatio(identity_explainer).compute(test_activations)

print(f"MSE: {round(mse, 3)}, FID: {round(fid, 3)}, Sparsity: {round(sparsity, 3)}, Sparsity ratio: {round(ratio, 3)}")

MSE: 183.787, FID: 0.009, Sparsity: 0.011, Sparsity ratio: 0.0
MSE: 0.0, FID: 0.0, Sparsity: 0.999, Sparsity ratio: 0.001


### 6.2 💭 Evaluate the concepts-interpretations from the [fourth step](#interpret)

In [14]:
# Work in progress, coming soon

## 7. Reusing the splitter model <a class="anchor" id="interface"></a>

Setting `llm_interface=None` uses the generation model already available in the splitter, avoiding another model load.

In [15]:
# No additional LLM needs to be loaded for concept labeling.

In [16]:
interpretation_method = LLMLabels(
    concept_explainer=concept_explainer,
    llm_interface=None,  # Reuse the generation model held by the splitter.
    k_examples=20,  # number of examples in each concept
    k_context=5,  # number of tokens before and after the maximally activating one to give context
    concept_encoding_batch_size=concept_model_batch_size,
)

# compute the labels for an arbitrary subset of the concepts
interpretations = interpretation_method.interpret(
    inputs=imdb,
    latent_activations=activations,
    concepts_indices=list(range(10)),  # we could put `"all"` but that would take lon and cost a bit through the API
)

for concept_id, label in interpretations.items():
    print(f"Concept {concept_id}: {label if label is not None else 'None'}")

Concept 0: Ġpeople
Concept 1: DVD
Concept 2: you
Concept 3: Ġjob
Concept 4: well
Concept 5: OK
Concept 6: Ġdo
Concept 7: Razz
Concept 8: Oscar
Concept 9: reading
